# Data merge: align prescan, in-scanner, and trait questionnaires

This notebook walks through merging three data sources into a single trial-level table:

1. **alldata.csv** — one row per scanner trial. Has the in-scanner reminder phrase, probe ratings (1–6), and other run-level data.
2. **Prescan thought interview** (Qualtrics CSV) — one row per (subject, thought) with thought ratings on 36 items. We compute `dyscontrol_imputed` as the mean of the repetitive, intrusive, and uncontrollable items.
3. **Trait baseline questionnaire** (Qualtrics CSV) — one row per subject with PTQ-15 and PSWQ-16 trait scores plus an attention check.

## Why we re-do this from scratch

The alldata.csv file already has prescan-derived columns added by an upstream R script using `fuzzyjoin`. Two problems with that:

- Fuzzy edit-distance matching gave **incorrect matches** in some cases (e.g., `walk` and `work` for sub-329 got swapped).
- The merge produced **duplicate rows** for subjects whose thoughts had similar reminders (e.g., sub-356 has every row duplicated).

Our approach: build a **strict** (exact-match) merge on normalized reminder phrases, with an explicit **manual_corrections.json** file listing the cases where the alldata and prescan reminders genuinely differ (typos, missing words, etc.). After applying the corrections, every alldata row should have exactly one matching prescan row.

## Outputs

- `derivatives/trial_level_merged.csv`: clean trial-level table for downstream analyses.

## Validation strategy

After merging, we sanity-check by:
1. Verifying merge counts match expectation (1566 trials = 58 subj × 27 trials, minus a few missing runs).
2. Pulling random (subject, thought) pairs and verifying the merged dyscontrol matches the source prescan CSV directly.
3. Verifying probe-dyscontrol correlations within subjects are positive (sanity for the merge: trial-level probes should track within-subject dyscontrol if they're correctly aligned).

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# import the loader modules

from config import (
    BASE, ALLDATA_PATH, PRESCAN_PATH, QUALTRICS_PATH,
    MANUAL_CORRECTIONS_PATH, DERIVATIVES_DIR, TRIAL_LEVEL_CSV,
    EXCLUDE_SUBJECTS,
)
from alldata_loader import load_alldata
from questionnaire_loader import (
    load_prescan_thought_ratings,
    load_trait_scores,
    normalize_thought_key,
)
from merge import (
    load_manual_corrections,
    apply_manual_corrections,
    strict_merge_prescan,
    final_dedup,
)

# Make derivatives directory if it doesn't exist
DERIVATIVES_DIR.mkdir(parents=True, exist_ok=True)

print('Paths:')
print(f'  ALLDATA_PATH:    {ALLDATA_PATH}')
print(f'  PRESCAN_PATH:    {PRESCAN_PATH}')
print(f'  QUALTRICS_PATH:  {QUALTRICS_PATH}')
print(f'  Output dir:      {DERIVATIVES_DIR}')

## Step 1: Load alldata (trial-level in-scanner data)

Each row is one scanner trial. The loader:
- Recombines rows where unquoted commas in the free-text Thought.x/Thought.y fields broke pandas parsing
- Drops the upstream R-fuzzy-merge prescan columns (we re-derive these later)
- Normalizes the reminder phrase to `thought_key` (lowercase, punctuation→spaces, whitespace collapsed)

In [ ]:
alldata = load_alldata(ALLDATA_PATH)
print(f'\nDataFrame shape: {alldata.shape}')
print(f'Subjects: {sorted(alldata["subject"].unique())}')
print(f'Runs per subject:')
print(alldata.groupby('subject')['run'].nunique().describe().round(1))
print(f'\nSample rows:')
alldata.head()

## Step 2: Load prescan thought ratings

One row per (subject, thought). We extract the 3 dyscontrol items (repetitive, intrusive, uncontrollable) from each thought block and compute their mean (NaN→0 imputed).

Note: the Qualtrics export has 9 thought blocks per subject. Each block has 36 items, but the first block (block 1) has a slightly different structure due to an emotion item gap. The loader handles this via integer column-position lookup.

In [ ]:
prescan = load_prescan_thought_ratings(PRESCAN_PATH)
print(f'\nDataFrame shape: {prescan.shape}')
print(f'\nDyscontrol distribution:')
print(prescan['dyscontrol_imputed'].describe().round(2))
print(f'\nSample rows:')
prescan.head(10)

## Step 3: Load trait scores (PTQ, PSWQ)

One row per subject. PTQ-15 + PSWQ-16. PTQ has an attention check item (PTQ_16) that should evaluate to ≥3 for the subject's data to be trusted.

In [ ]:
traits = load_trait_scores(QUALTRICS_PATH)
print(f'\nDataFrame shape: {traits.shape}')
print(f'\nSample rows:')
traits.head()

### Spot-check 2: verify trait scoring on 3 random subjects

For 3 random subjects, list the PTQ and PSWQ raw items and the computed totals. We check that the total = sum (with reverse-coding for PSWQ items 1, 3, 8, 10, 11).

In [ ]:
# Sanity check that PTQ and PSWQ totals are in the expected ranges:
# PTQ-15: 0-60 (each item 0-4)
# PSWQ-16: 16-80 (each item 1-5, sum after reverse-coding)
print('Trait score distributions:')
valid = traits.dropna(subset=['ptq_total', 'pswq_total'])
print(f'  PTQ total: M={valid["ptq_total"].mean():.1f}, '
      f'SD={valid["ptq_total"].std():.1f}, '
      f'range {valid["ptq_total"].min():.0f}-{valid["ptq_total"].max():.0f} '
      f'(expected 0-60)')
print(f'  PSWQ total: M={valid["pswq_total"].mean():.1f}, '
      f'SD={valid["pswq_total"].std():.1f}, '
      f'range {valid["pswq_total"].min():.0f}-{valid["pswq_total"].max():.0f} '
      f'(expected 16-80)')

# Sanity: PTQ and PSWQ should be positively correlated (both measure trait worry/rumination)
from scipy.stats import pearsonr
r, p = pearsonr(valid['ptq_total'], valid['pswq_total'])
print(f'\nPTQ-PSWQ correlation: r={r:.3f}, p={p:.4f}')
print(f'  (expected r > 0.5 if both measures are valid)')

# Attention check pass rate
n_pass = traits['ptq_attention_pass'].sum()
print(f'\nPTQ attention check passed: {n_pass}/{len(traits)} subjects')

## Step 4: Apply manual corrections to alldata

Some alldata thought_keys don't exact-match the prescan thought_keys due to:
- Typos that differ between the two sources (e.g., 'weather' vs 'weater')
- Missing words (e.g., 'family mom dad' vs 'family mom and dad')
- Apostrophes (handled by `normalize_thought_key`, but some edge cases remain)
- Synonyms (e.g., 'shop' vs 'cafe' for the same concept)

These corrections are listed in `manual_corrections.json` and applied here. After this, every alldata row should match a prescan row exactly.

In [ ]:
corrections = load_manual_corrections(MANUAL_CORRECTIONS_PATH)
print(f'Loaded {len(corrections)} manual corrections')
print(f'\nFirst few:')
for c in corrections[:5]:
    print(f"  sub-{c['subject']}: '{c['from']}' -> '{c['to']}'  ({c['note']})")

alldata_corrected = apply_manual_corrections(alldata, corrections)

## Step 5: Strict merge alldata + prescan

Inner-join on (subject, thought_key). If everything is right, **0 unmatched rows**.

In [ ]:
matched, unmatched = strict_merge_prescan(alldata_corrected, prescan)

if len(unmatched) > 0:
    print('\n*** WARNING: unmatched rows present ***')
    print('Either add manual corrections to manual_corrections.json,')
    print('or investigate the source data discrepancy.')

## Step 6: Final dedup

Drop row-level duplicates on (subject, run, ThoughtNumber, thought_key).

This catches the sub-356 issue where the source alldata.csv had every row duplicated. After this, we have one row per scanner trial.

In [ ]:
trial_level = final_dedup(matched)

# Apply EXCLUDE_SUBJECTS
before = len(trial_level)
trial_level = trial_level[~trial_level['subject'].isin(EXCLUDE_SUBJECTS)].reset_index(drop=True)
print(f'\nApplied EXCLUDE_SUBJECTS={EXCLUDE_SUBJECTS}: '
      f'{before} -> {len(trial_level)} rows')

# Merge in trait scores (left join, since some subjects don't have trait data)
trial_level = trial_level.merge(traits, on='subject', how='left')
print(f'\nFinal shape (with traits merged): {trial_level.shape}')

n_with_traits = trial_level['ptq_total'].notna().any() and len(
    trial_level.dropna(subset=['ptq_total'])['subject'].unique()
)
print(f'Subjects with trait data: {n_with_traits}/{trial_level["subject"].nunique()}')

## Spot-check 3: random trial-level rows

Pull 5 random rows from the merged data and manually verify against the source CSVs. For each, we want to check:
- The (subject, thought_key) appears in alldata
- The same (subject, thought_key) appears in prescan with matching dyscontrol
- The trait scores match what the trait CSV has for that subject

In [ ]:
np.random.seed(42)
spot_sample = trial_level.sample(5, random_state=42).reset_index(drop=True)

for _, row in spot_sample.iterrows():
    sub = int(row['subject'])
    thought = row['thought_key']
    print(f"\n--- sub-{sub} | thought='{thought}' ---")
    
    print(f"  In alldata (corrected): "
          f"{((alldata_corrected['subject']==sub) & (alldata_corrected['thought_key']==thought)).sum()} rows")
    print(f"  In prescan: "
          f"{((prescan['subject']==sub) & (prescan['thought_key']==thought)).sum()} rows")
    print(f"  In traits:  {(traits['subject']==sub).sum()} rows")
    
    # Prescan dyscontrol -- should match merged
    prescan_match = prescan[(prescan['subject']==sub) & (prescan['thought_key']==thought)]
    print(f'\n  DYSCONTROL:')
    if len(prescan_match) > 0:
        print(f"    prescan dyscontrol_imputed: {prescan_match['dyscontrol_imputed'].iloc[0]:.3f}")
    print(f"    merged dyscontrol_imputed:  {row['dyscontrol_imputed']:.3f}")
    
    # Trait scores -- should match raw traits values
    traits_match = traits[traits['subject']==sub]
    print(f'\n  PTQ:')
    if len(traits_match) > 0:
        print(f"    raw traits ptq_total: {traits_match['ptq_total'].iloc[0]}")
    print(f"    merged ptq_total:     {row.get('ptq_total', 'NaN')}")
    print(f'\n  PSWQ:')
    if len(traits_match) > 0:
        print(f"    raw traits pswq_total: {traits_match['pswq_total'].iloc[0]}")
    print(f"    merged pswq_total:     {row.get('pswq_total', 'NaN')}")

### Spot-check 4: known-canary trials (sub-329 work/walk)

Sub-329 had two reminder phrases — 'work' (high dyscontrol) and 'walk' (low dyscontrol) — that got swapped by the upstream R fuzzy-match. After our strict merge, 'work' should map to high dyscontrol (~8) and 'walk' should map to low (~0).

In [ ]:
for thought in ['work', 'walk']:
    rows = trial_level[
        (trial_level['subject']==329) & (trial_level['thought_key']==thought)
    ]
    if len(rows) > 0:
        print(f"sub-329 '{thought}': {len(rows)} trials, "
              f"dyscontrol={rows['dyscontrol_imputed'].iloc[0]:.3f}")
    else:
        print(f"sub-329 '{thought}': no rows found")

In [ ]:
dupes = (prescan.groupby(['subject', 'thought_key']).size().reset_index(name='n')
         .query('n > 1'))
print(f'(subject, thought_key) pairs duplicated in prescan: {len(dupes)}')
if len(dupes) > 0:
    print(dupes.to_string(index=False))

In [ ]:
print('sub-356 prescan duplicates - do dyscontrol values match between duplicates?')
sub356_prescan = prescan[prescan['subject'] == 356]
print(f'\nTotal sub-356 prescan rows: {len(sub356_prescan)}')
print()

# For each duplicated thought, show both rows side-by-side
for thought in sub356_prescan['thought_key'].unique():
    rows = sub356_prescan[sub356_prescan['thought_key'] == thought]
    if len(rows) > 1:
        d_values = rows['dyscontrol_imputed'].tolist()
        positions = rows['thought_position'].tolist()
        match = '✓ same' if len(set(d_values)) == 1 else '✗ DIFFER'
        print(f'  {thought:25s}: positions={positions}, '
              f'dyscontrol={[f"{d:.2f}" for d in d_values]}  {match}')

# Also check: are ALL the ratings the same, not just dyscontrol?
print('\nSpot check on one thought - show full rating arrays:')
example = sub356_prescan[sub356_prescan['thought_key'] == 'friendships']
quality_cols = [f'quality_{i}' for i in range(1, 9)]
print(example[['thought_position'] + quality_cols].to_string(index=False))

### Spot-check 5: probe-dyscontrol within-subject correlations

If the merge is correct, the probe ratings (1, 4, 5, 6) — which ask about how repetitive/intrusive the thought was *during the scan* — should positively correlate with the prescan dyscontrol within each subject. This is a sanity check on alignment: if dyscontrol values are scrambled across thoughts (e.g., from a bad merge), correlations would be near zero or negative.

Expected: probe1, probe4, probe5 should have positive within-subject correlations with dyscontrol_imputed (mean r > 0.3, most subjects positive).

In [ ]:
from scipy.stats import pearsonr

probe_cols = [f'probe{i}' for i in range(1, 7)]
print('Within-subject correlations between dyscontrol_imputed and probes:')
print(f"{'probe':6s}  {'mean r':>8s}  {'%pos':>5s}  {'n_subj':>6s}")
print('-' * 40)
for probe in probe_cols:
    if probe not in trial_level.columns:
        continue
    rs = []
    for sub, g in trial_level.groupby('subject'):
        valid = g.dropna(subset=['dyscontrol_imputed', probe])
        if len(valid) < 5 or valid[probe].std() == 0 or valid['dyscontrol_imputed'].std() == 0:
            continue
        r, _ = pearsonr(valid['dyscontrol_imputed'], pd.to_numeric(valid[probe], errors='coerce'))
        if not np.isnan(r):
            rs.append(r)
    pct_pos = 100 * np.mean(np.array(rs) > 0)
    print(f'{probe:6s}  {np.mean(rs):>+8.3f}  {pct_pos:>4.0f}%  {len(rs):>6d}')

## Step 7: Save trial-level merged data

In [ ]:
trial_level.to_csv(TRIAL_LEVEL_CSV, index=False)
print(f'Saved: {TRIAL_LEVEL_CSV}')
print(f'  {len(trial_level)} rows, {trial_level["subject"].nunique()} subjects, '
      f'{trial_level.shape[1]} columns')